In [ ]:


from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

Prompts 

In [21]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a strict, no-fluff AI tutor."),
    ("human", "explain {topic}")
])

messages = prompt.format_messages(topic="RAG (Retrieval-Augmented Generation")
print(messages)

response = model.invoke(messages)
print(response.content)

[SystemMessage(content='you are a strict, no-fluff AI tutor.', additional_kwargs={}, response_metadata={}), HumanMessage(content='explain RAG (Retrieval-Augmented Generation', additional_kwargs={}, response_metadata={})]
[{'type': 'text', 'text': '**Retrieval-Augmented Generation (RAG)** is an architecture used to optimize Large Language Model (LLM) outputs by referencing a specific, authoritative knowledge base outside of its training data.\n\n### 1. The Core Problem\nLLMs have two primary limitations:\n*   **Knowledge Cutoff:** They only know what they were trained on up to a certain date.\n*   **Hallucination:** They generate confident but false information when they lack specific facts.\n\nRAG solves this by providing the model with a "closed-book exam" approach where it can look up facts in real-time.\n\n### 2. The RAG Workflow\nThe process follows three distinct steps:\n\n1.  **Retrieval:** When a user asks a question, the system searches an external data source (usually a Vector

Loaders 


In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("notes.txt")
docs = loader.load()
print(docs[0].page_content)   # full file content
print(docs[0].metadata)       # {'source': 'notes.txt'}

In [ ]:
from langchain_community.document_loaders import PyPDFLoaders

loader = PyPDFLoader("report.pdf")
docs = loader.load()
print(len(docs))              # one Document PER PAGE
print(docs[2].metadata)       # {'source': 'report.pdf', 'page': 2}

ModuleNotFoundError: No module named 'langchain_community.documents_loaders'

Chains 

chain = prompt | model | parser

# Under the hood, chain.invoke(input) does roughly:
step1 = prompt.invoke(input)      # → list of messages
step2 = model.invoke(step1)       # → AIMessage
step3 = parser.invoke(step2)      # → plain string


In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence.")
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({"topic": "gradient descent"})
print(result)

Gradient descent is an iterative optimization algorithm used to minimize a function by repeatedly moving in the direction of steepest descent as defined by the negative of the gradient.


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loader
docs = TextLoader("sample.txt").load()

# Split
chunks = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10).split_documents(docs)

# Prompt
summarize_prompt = ChatPromptTemplate.from_template("Summarize in one line: {text}")

# Chain
summarize_chain = summarize_prompt | model | parser

# Run the chain over every chunk the loader produced
for chunk in chunks:
    print(summarize_chain.invoke({"text": chunk.page_content}))